# **Lab 1: Preprocessing & Data Wrangling**

**Course**: **INS-605: Data Analysis** <br>
**Lecturer**: **Sothea HAS, PhD**

-----

**Objective:** In this lab, we will work with real-world product review data (`Amazon Product Reviews`) to practice key data wrangling techniques discussed in the lecture: inspecting raw data, handling missing values, converting data types, feature transformation, grouping/aggregation, combining datasets, and reshaping data.

> The notebook of the lab can be downloaded from [Lab1: Proprocessing & Data Wrangling](https://hassothea.github.io/AUPP_Data_Analysis_II/Labs/Lab1/01-data-wrangling.ipynb){target="_blank"}.

-----


- Student name: Laimeng EUNG
- ID: 2024440

> **Submission:** You must do the followings
> 
> - Name your file as: `Lab1_name.ipynb`
> - Submit this jupyter notebook file (`Lab1_name.ipynb`) to Canvas, NOT THE COLAB LINK!


-----

## **1. Importing and Inspecting the Data**

We will explore the **Amazon Product Reviews** dataset from Kaggle (`arhamrumi/amazon-product-reviews`). Let's download and load the dataset into pandas.

In [1]:
#| echo: ture
#| code-fold: true

# %pip install kagglehub

import kagglehub
import pandas as pd
import os

# Download latest version
path = kagglehub.dataset_download("arhamrumi/amazon-product-reviews")

# Load data (locate the CSV file in the downloaded path)
file_name = [f for f in os.listdir(path) if f.endswith('.csv')][0]
data = pd.read_csv(f"{path}/{file_name}")
data.head()

c:\Users\eungl\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.4.0) or chardet (7.4.3)/charset_normalizer (3.4.1) doesn't match a supported version!
  warnings.warn(


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


-------

### **1.1. Overview of the data**

Before cleaning, inspect the structure of the dataset and answer the following:

**A.** Check the dimension of the dataset (`shape`). How many rows and columns are there?

**B.** What are the data types of each column (`dtypes`)? Identify columns whose data types do not match their intended semantic meaning.

**C.** Check for missing values (`isna().sum()`). Which columns contain `NaN` values, and how many are missing in each column?

**D.** Are there any duplicated rows in this dataset? Calculate the total number of duplicates using `.duplicated().sum()`.

**E.** Check the number of unique values in each column (`nunique()`). 

- **E.1.** Compare the number of unique `UserId`s to unique `ProfileName`s. Why do you think we observed such numbers?
- **E.2.** Do missing names in `ProfileName` potentially share a `UserId` with non-missing rows? This will guide our imputation strategy.
  
**F.** Check if missing `ProfileName`s are related to missing `Summary`s.

In [2]:
# Let's find out!
# A.
data.shape

(568454, 10)

In [3]:
# B. Type
data.dtypes

Id                         int64
ProductId                 object
UserId                    object
ProfileName               object
HelpfulnessNumerator       int64
HelpfulnessDenominator     int64
Score                      int64
Time                       int64
Summary                   object
Text                      object
dtype: object

`Time` has the wrong data type. It should be `date` instead of `int64`

In [4]:
# Missing values
data.isna().sum()

Id                         0
ProductId                  0
UserId                     0
ProfileName               26
HelpfulnessNumerator       0
HelpfulnessDenominator     0
Score                      0
Time                       0
Summary                   27
Text                       0
dtype: int64

`ProfileName` and `Summary` has 26 and 27 missing values, respectively.

In [5]:
# D. Check for duplicated values
data.duplicated().sum()

np.int64(0)

There are 0 duplicate row.

**E.** Check the number of unique values in each column (`nunique()`). 

- **E.1.** Compare the number of unique `UserId`s to unique `ProfileName`s. Why do you think we observed such numbers?
- **E.2.** Do missing names in `ProfileName` potentially share a `UserId` with non-missing rows? This will guide our imputation strategy.
  
**F.** Check if missing `ProfileName`s are related to missing `Summary`s.

In [6]:
# E. Check for unique values in each column
data.nunique()

Id                        568454
ProductId                  74258
UserId                    256059
ProfileName               218415
HelpfulnessNumerator         231
HelpfulnessDenominator       234
Score                          5
Time                        3168
Summary                   295742
Text                      393579
dtype: int64

In [7]:
# E.1: compare the unique UserID to ProfileName
# Check for UserId with missing profile name
missing_names = data[['UserId', 'ProfileName']].query("ProfileName.isna()")
missing_names

,UserId,ProfileName
10616,A1LSYR30XW7CFT,NaN
25509,A36BVYD0NT7Z0F,NaN
38874,A36BVYD0NT7Z0F,NaN
47923,A2DEHJJIEAPPBF,NaN
49800,A2LYFY32LXQDON,NaN
67077,A2P0P67Y55SNOX,NaN
106550,A1P500QXEG3IUZ,NaN
121819,A34F9B57SYQ5XR,NaN
125452,A34F9B57SYQ5XR,NaN
137613,AGT3BYX5P9SLH,NaN


In [8]:
missing_names.shape

(26, 2)

In [9]:
# missing_names.query('ProfileName.isna()')
# Dropping NAN of either UserId or ProfileName
# .first() flatten each UserId group into a single ProfileName string so Pandas can format the result into a key-value dictionary.
name_map = data[['UserId', 'ProfileName']].dropna().groupby('UserId').first().to_dict()
name_map['ProfileName']

{'#oc-R103C0QSV1DF5E': 'C',
 '#oc-R109MU5OBBZ59U': 'AayGee',
 '#oc-R10LFEMQEW6QGZ': 'Julie',
 '#oc-R10LT57ZGIB140': 'dipr',
 '#oc-R10UA029WVWIUI': 'Kim D',
 '#oc-R115TNMSPFT9I7': 'Breyton',
 '#oc-R119LM8D59ZW8Y': 'Lee Mellott "SEOmenu2go"',
 '#oc-R11D9D7SHXIJB9': 'Louis E. Emory "hoppy"',
 '#oc-R11D9LKDAN5NQJ': 'Melissa Korducavich',
 '#oc-R11DNU2NBKQ23Z': 'Kim Cieszykowski',
 '#oc-R11O5J5ZVQE25C': 'Penguin Chick',
 '#oc-R11PW3CFBB4BEP': 'Thomas "mrpalmer420"',
 '#oc-R11T1PHWNO7KEZ': 'David Zeitler',
 '#oc-R120LO6OLNDPCG': 'cjl',
 '#oc-R12KPBODL2B5ZD': 'Christopher P. Presta',
 '#oc-R12MGTQS5KZZRV': 'SKY2110 "SKY2110"',
 '#oc-R12N3533IO3B79': 'kat',
 '#oc-R13EBF129DBX88': 'mary',
 '#oc-R13NNUL4EKL4FL': 'N. Chernyavskaya',
 '#oc-R13X3YIJ6GLT0C': 'P. McCoy "Friend of Bill W. and Dr. Bob"',
 '#oc-R1493EDK6REQ02': 'Robud48',
 '#oc-R149FDXLRARCWJ': 'Alicia Johnson',
 '#oc-R14VLCY75KIBSB': 'steve "Steve"',
 '#oc-R14ZSRYW2YB41B': 'A. Crafton',
 '#oc-R14ZUK54VMOGJS': 'Dave',
 '#oc-R1522DF2LUL4

In [10]:
missing_names['UserId'].map(name_map)

10616     NaN
25509     NaN
38874     NaN
47923     NaN
49800     NaN
67077     NaN
106550    NaN
121819    NaN
125452    NaN
137613    NaN
163191    NaN
172462    NaN
211846    NaN
268648    NaN
297275    NaN
306751    NaN
331647    NaN
358674    NaN
431598    NaN
440825    NaN
461668    NaN
490412    NaN
491728    NaN
515436    NaN
517676    NaN
560446    NaN
Name: UserId, dtype: object

In [11]:
# Profile name and Missing summary
data[['ProfileName', 'Summary']].query("ProfileName.isna() or Summary.isna()").shape

data[data["ProfileName"].isna() & data["Summary"].isna()]

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text


They are not related!

-------

## **2. Data Cleaning**

Now that you have identified the issues in raw data, let's clean them step-by-step.

**F. Datetime Conversion:** Convert the `Time` column (Unix timestamp in seconds) into a proper pandas `datetime` format using `pd.to_datetime(..., unit='s')`.

**G. Handling Missing Text Values (Smart Imputation):** 
- **G1. Recovering Profile Names:** Since users might have multiple reviews, a missing `ProfileName` might have a known `ProfileName` under the same `UserId` in another row. Check if there are such cases, therefore use this information to impute the missing values.
- **G2. Fallback Imputation:** Fill any *remaining* missing values in `ProfileName` with `'Unknown'`, and fill missing values in `Summary` with `'No Summary'`.

**H. Duplicates:** If any duplicated rows exist, drop them using `.drop_duplicates()` and verify the shape of your DataFrame afterward.

In [12]:
# To do
# F. convert Time to datetime
data["Time"] = pd.to_datetime(data["Time"], unit='s')
data.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,2011-04-27,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,2012-09-07,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,2008-08-18,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,2011-06-13,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,2012-10-21,Great taffy,Great taffy at a great price. There was a wid...


In [13]:
# G1. Recovering Profile names
# Check if there is a known ProfileName under the same UserId

# data.where(data.query())

# G2. Feedback Imputation
data["ProfileName"] = data["ProfileName"].fillna("Unknown")
data["Summary"] = data["Summary"].fillna("No Summary")

In [14]:
# H. Duplicates
# Drop Id column first because Id are all unique
print("Dropping Id column")
# data = data.drop(columns=['Id'])

# Check for all duplicate rows
# data[data.duplicated(keep=False)].sort_values(by=["UserId", "Text"])

print("Duplicated rows: ", len(data[data.duplicated()]))


print("Before dropping duplicates", data.shape)
data = data.drop_duplicates()
print("After dropping duplicates", data.shape)

Dropping Id column
Duplicated rows:  0
Before dropping duplicates (568454, 10)
After dropping duplicates (568454, 10)


-------

## **3. Feature Transformation**

Feature engineering helps turn raw fields into more useful analytical features.

**I. Rating Classification (`Status`):** Create a new column named `Status` based on the review rating (`Score`): 

- `'High'` if `Score >= 4`
- `'Medium'` if `Score == 3`
- `'Low'` if `Score <= 2`

**J. Temporal Extraction (`Year`):** Extract the year from your converted `Time` column and store it in a new column called `Year`.

**K. Metric Calculation (`HelpfulnessRatio`):** Create a column `HelpfulnessRatio` by dividing `HelpfulnessNumerator` by `HelpfulnessDenominator`. Handle division-by-zero cases (e.g., set to 0 when `HelpfulnessDenominator == 0`).

In [15]:
data.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,2011-04-27,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,2012-09-07,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,2008-08-18,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,2011-06-13,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,2012-10-21,Great taffy,Great taffy at a great price. There was a wid...


In [16]:
# To do
# I. Rating Classification
bins = [0, 2, 3, 5]
labels = ["Low", "Medium", "High"]

data["Status"] = pd.cut(data["Score"], bins=bins, labels=labels)
data.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,Status
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,2011-04-27,Good Quality Dog Food,I have bought several of the Vitality canned d...,High
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,2012-09-07,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...,Low
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,2008-08-18,"""Delight"" says it all",This is a confection that has been around a fe...,High
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,2011-06-13,Cough Medicine,If you are looking for the secret ingredient i...,Low
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,2012-10-21,Great taffy,Great taffy at a great price. There was a wid...,High


In [17]:
# J. Extract year from Time
data["Year"] = data["Time"].dt.year

data.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,Status,Year
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,2011-04-27,Good Quality Dog Food,I have bought several of the Vitality canned d...,High,2011
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,2012-09-07,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...,Low,2012
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,2008-08-18,"""Delight"" says it all",This is a confection that has been around a fe...,High,2008
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,2011-06-13,Cough Medicine,If you are looking for the secret ingredient i...,Low,2011
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,2012-10-21,Great taffy,Great taffy at a great price. There was a wid...,High,2012


In [18]:
import numpy as np
# K. Create HelpfulnessRatio column (HelpfulNum / HelpfulDen); handle division-by-zero (set to 0)
data["HelpfulnessRatio"] = np.where(
    data["HelpfulnessDenominator"] > 0,
    data["HelpfulnessNumerator"] / data["HelpfulnessDenominator"],
    0
)

data.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,Status,Year,HelpfulnessRatio
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,2011-04-27,Good Quality Dog Food,I have bought several of the Vitality canned d...,High,2011,1.0
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,2012-09-07,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...,Low,2012,0.0
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,2008-08-18,"""Delight"" says it all",This is a confection that has been around a fe...,High,2008,1.0
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,2011-06-13,Cough Medicine,If you are looking for the secret ingredient i...,Low,2011,1.0
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,2012-10-21,Great taffy,Great taffy at a great price. There was a wid...,High,2012,0.0


In [26]:
# Export data as a csv file to use Lab 2
PATH = "datasets/cleaned_amazon_product_reviews.csv"

data.to_csv(PATH, index=False)
print("Data saved to", PATH)

Data saved to datasets/cleaned_amazon_product_reviews.csv


-------

## **4. Grouping & Aggregation**

**L. Most Reviewed Products:** Group by `ProductId`. Find the **top 5 products** with the highest number of reviews.

**M. User Activity:** Group by `UserId`. Find the **top 5 most active reviewers** (users with the most reviews) and calculate their average review `Score`.

**N. Product Metrics:** For the top 5 products identified in Question **L**, use `.groupby()` and `.agg()` to compute both the average `Score` and total count of reviews.

In [19]:
# To do
# L. Most Reviewed products Group by ProductId
print("Top 5 products with the highest number of reviews")
data.groupby("ProductId").size().sort_values(ascending=False).head()

Top 5 products with the highest number of reviews


ProductId
B007JFMH8M    913
B002QWP8H0    632
B002QWHJOU    632
B002QWP89S    632
B0026RQTGE    632
dtype: int64

In [20]:
# M. user Activity group by UserId & Calculate their average review Score
print("Top 5 most active reviews")

user_activity = data.groupby("UserId").agg(
    review_count=("Score", 'size'),
    average_score=("Score", "mean")
)
user_activity.sort_values(by="review_count",ascending=False).head()

Top 5 most active reviews


,review_count,average_score
UserId,,
A3OXHLG6DIBRW8,448,4.535714
A1YUL9PCJR3JTY,421,4.494062
AY12DBB0U420B,389,4.647815
A281NPSIMI1C2R,365,4.841096
A1Z54EM24Y40LL,256,4.453125


**N. Product Metrics:** For the top 5 products identified in Question **L**, use `.groupby()` and `.agg()` to compute both the average `Score` and total count of reviews.

In [21]:
# N. compute avr score and total count of reviews for top 5 products
most_reviewed_products = data.groupby("ProductId").agg(
    review_count=("Score", "size"), # can be any field btw, not just Score
    average_count=("Score", "mean")
).round(2)  # we can add .round after the whole agg func

top_5_most_reviewed_products = most_reviewed_products.sort_values(by="review_count", ascending=False).head().reset_index()  # reset ProductId from index into column label
top_5_most_reviewed_products

,ProductId,review_count,average_count
0,B007JFMH8M,913,4.58
1,B002QWP8H0,632,4.59
2,B002QWHJOU,632,4.59
3,B002QWP89S,632,4.59
4,B0026RQTGE,632,4.59


-------

## **5. Combining and Reshaping Data**

**O. Combining Datasets (`pd.merge`):** Create a small lookup DataFrame named `product_catalog` containing:

- `ProductId`: The top 5 product IDs from Question **L**
- `Category`: Assigned categories (e.g., `'Beverages'`, `'Snacks'`, `'Gourmet'`, `'Pet Supplies'`, `'Organic'`)

Perform an **inner join** (`pd.merge`) between your aggregated product metrics from Question **N** and `product_catalog`.

**P. Reshaping Data (`pd.pivot_table`):** Create a pivot table using `pd.pivot_table()` to summarize review volume:

- **Index (Rows):** `Year`
- **Columns:** `Status` (`High`, `Medium`, `Low`)
- **Values:** Count of reviews (`Score`)
- **Mean:** average score per category of `Status`.
- Fill missing cells with 0.

**Q.** How does the platform perform over the years?

In [22]:
# To do
# O. Combining Datasets (pd.merge)
    # Create a lookup df product_catalog containign top 5 products and 5 categories
product_catalog = pd.DataFrame(
    {
        'ProductId': top_5_most_reviewed_products['ProductId'],
        'Category': ['Beverages', 'Snacks', 'Gourmet', 'Pet Supplies', 'Organic']
    }
)
product_catalog

merged_product = pd.merge(top_5_most_reviewed_products, product_catalog, how='inner')
merged_product

,ProductId,review_count,average_count,Category
0,B007JFMH8M,913,4.58,Beverages
1,B002QWP8H0,632,4.59,Snacks
2,B002QWHJOU,632,4.59,Gourmet
3,B002QWP89S,632,4.59,Pet Supplies
4,B0026RQTGE,632,4.59,Organic


In [23]:
# P. Pivot table to summarize review volume
pivot_table = pd.pivot_table(
    data,
    index='Year',
    columns='Status',
    values='Score',
    aggfunc=['count', 'mean'],
    fill_value=0
).round(2)

pivot_table

C:\Users\eungl\AppData\Local\Temp\ipykernel_17876\1306149491.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(
C:\Users\eungl\AppData\Local\Temp\ipykernel_17876\1306149491.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(


count                 mean             
Status    Low Medium    High   Low Medium  High
Year                                           
1999        0      0       6  0.00    0.0  5.00
2000        3      0      29  1.00    0.0  4.90
2001        4      1       8  1.25    3.0  4.75
2002        3      0      70  2.00    0.0  4.83
2003       12      8     112  1.17    3.0  4.76
2004       58     18     485  1.17    3.0  4.83
2005      142     53    1140  1.58    3.0  4.87
2006      783    428    5460  1.33    3.0  4.84
2007     2164   1326   18810  1.42    3.0  4.83
2008     3542   2171   28450  1.36    3.0  4.82
2009     6496   3762   45068  1.40    3.0  4.83
2010    12317   6248   67319  1.36    3.0  4.83
2011    24937  13022  125340  1.36    3.0  4.82
2012    31576  15603  151480  1.35    3.0  4.81

In [24]:
# **Q.** How does the platform perform over the years?
yearly_data = data.groupby('Year')['Score'].agg(['count', 'mean'])

yearly_data

,count,mean
Year,,
1999,6,5.000000
2000,32,4.531250
2001,13,3.538462
2002,73,4.712329
2003,132,4.325758
2004,561,4.392157
2005,1335,4.445693
2006,6671,4.313446
2007,22300,4.391345


**Q.** How does the platform perform over the years?
> Over the years, the review counts has grown gradually from 6 in 1999 to almost 200,000 reviews in 2012. With the increase of review counts, the average review score also seem to gradually decrease from perfect 5 score in 1999 to dropping as low as 3.5 in 2001, but bounced back the next year up to 4.7, and slowly decrease until it reached 4.1 in 2012.

# **Further Reading**

- `Pandas` Data Wrangling Cheat Sheet: https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf

- Chapter 6-8, Python for Data Analysis by Wes McKinney: https://wesmckinney.com/book/

- `Pandas` Apply, Transform, Map: https://pandas.pydata.org/docs/user_guide/basics.html
